In [1]:
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.2.3
!pip install -q langchain-community==0.4.1
!pip install -q langchain-core==1.2.6
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0
!pip install -q rank-bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependenc

In [6]:
import os 
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

In [7]:
!wget -O buku_panduan_gen_ai.pdf "https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb"

--2026-05-26 05:55:50--  https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.google.com (drive.google.com)... 142.250.141.139, 142.250.141.102, 142.250.141.101, ...
Connecting to drive.google.com (drive.google.com)|142.250.141.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb [following]
--2026-05-26 05:55:51--  https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.137.132, 2607:f8b0:4023:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.137.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 655208 (640K) [application/octet-stream]
Saving to: ‘buku_panduan_gen_ai.pdf’

buku_panduan_gen_ai 100%[===================>] 639.85K  --.-KB/s    in 0.1s    

2026-05-26 05:55:52

In [8]:
file_path = "/content/buku_panduan_gen_ai.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

In [9]:
#Parent Chunking
paragraph_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

#Child Chunking
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [12]:
vectorstore = Chroma(
    collection_name="split_parents",
    embedding_function=embedding_model
)

/tmp/ipykernel_566/3269059110.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [13]:
docstore = InMemoryByteStore()

In [15]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=paragraph_splitter,
    search_type="similarity",
    search_kwargs={"k": 10}    
)

retriever.add_documents(documents)

In [16]:
query = "Framework apa yang dapat digunakan untuk memastikan pengguna GenAI yang efektif?"
relevant_docs = retriever.invoke(query)

for i, doc in enumerate(relevant_docs, start=1):
    print(f"--- Document {i} ---")
    print(doc.page_content)
    print()

--- Document 1 ---
17
3.3. T.U.C.E. Framework (Think, Use, Check, En-
hance)
Untuk memastikan penggunaan GenAI yang efektif dan bertanggung jawab, maha-
siswa dapat mengingat Framework berikut ini:
1.	 Think (Pikirkan Sebelum Menggunakan GenAI)
a.	 Tentukan tujuan penggunaan GenAI: Apakah GenAI benar-benar diperlukan?
b.	 Pastikan GenAI digunakan sebagai alat bantu, bukan pengganti pemikiran kritis.
c.	 Periksa aturan mata kuliah atau dosen terkait penggunaan GenAI dalam tugas.
2.	 Use (Gunakan GenAI dengan Bijak)
a.	 Pilih alat GenAI yang sesuai dengan kebutuhan (ChatGPT untuk teks, DALL-E 
untuk gambar, dll.).
b.	 Masukkan instruksi yang jelas dan spesifik agar hasilnya relevan.
c.	 Hindari memasukkan data pribadi atau informasi sensitif ke dalam GenAI.
3.	 Check (Periksa Hasil dan Verifikasi Informasi)
a.	 Cek keakuratan dan kredibilitas hasil GenAI dengan sumber akademik yang valid.
b.	 Pastikan tidak ada plagiarisme dan berikan atribusi jika diperlukan.
c.	 Evaluasi apakah GenAI m

In [17]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10

In [18]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever],
    weights=[0.5, 0.5]
)

In [19]:
query = "Apa sih itu GenAI detector"
hybrid_relevant_docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(hybrid_relevant_docs, start=1):
    print(f"--- Hybrid Document {i} ---")
    print(doc.page_content)
    print()

--- Hybrid Document 1 ---
31
BAB V: Tanya Jawab Seputar 
Penggunaan Generative AI Dalam Akademik
Bagian ini berisi pertanyaan umum yang sering diajukan mahasiswa terkait penggu-
naan GenAI dalam lingkungan akademik, serta batasan dan aturan spesifik yang ber-
laku.
1.	 Kapan mahasiswa boleh menggunakan GenAI dalam tugas akademik? 
Mahasiswa diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 GenAI digunakan sebagai alat bantu dalam brainstorming, merangkum, atau 
menyusun ide.
2.	 GenAI membantu mengoreksi tata bahasa dan meningkatkan kualitas tulisan.
3.	 Mahasiswa tetap memahami dan menyesuaikan hasil GenAI dengan analisis dan 
pemikiran mereka sendiri.
4.	 Sumber GenAI dicantumkan dengan format kutipan yang sesuai.
2.	 Kapan GenAI tidak boleh digunakan dalam tugas akademik?
Mahasiswa tidak diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 Dosen secara eksplisit melarang penggunaan GenAI dalam tugas tertentu.
2.	 Mahasiswa mengandalkan GenAI sepenuhnya tanpa m

In [21]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

compressor = CrossEncoderReranker(model=model, top_n=5)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

In [22]:
final_docs = compression_retriever.invoke(query)

for i, doc in enumerate(final_docs, start=1):
    print(f"--- Compressed Document {i} ---")
    print(doc.page_content)
    print()

--- Compressed Document 1 ---
25
BAB IV: Ruang Lingkup Integritas Akademik 
Dalam Penggunaan Generative AI
Penggunaan Generative AI (GenAI) dalam lingkungan akademik harus tetap menjun-
jung tinggi integritas akademik, yaitu prinsip kejujuran, tanggung jawab, dan etika da-
lam belajar dan berkarya. Untuk memastikan hal ini, diperlukan strategi yang mencak-
up pencegahan, pembinaan, dan penanggulangan terhadap penyalahgunaan AI dalam 
tugas akademik.
4.1. Pencegahan: Mencegah Penyalahgunaan Gen-
erative AI Sejak Awal
4.1.1. Memahami Kemampuan Dosen dalam Mengetahui Penggunaan Genera-
tive AI pada Mahasiswa
Mahasiswa perlu menyadari bahwa dosen memiliki berbagai cara untuk mendeteksi 
penggunaan GenAI dalam tugas akademik. Beberapa metode yang digunakan meliputi:
1.	 Penggunaan GenAI Detector: Dosen dapat menggunakan alat pendeteksi GenAI 
seperti Turnitin AI Detector, GPTZero, dan Originality.ai untuk mengidentifikasi 
teks yang dihasilkan oleh GenAI.
2.	 Analisis Gaya Tulisan: Dosen da

In [25]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "unsloth/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [26]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1000,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cuda:0


In [27]:
template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
 
Anda adalah asisten AI yang bertugas membantu pengguna.
Gunakan hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan.
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jujur bahwa Anda tidak mengetahui jawabannya dan jangan membuat asumsi atau jawaban tambahan.
Berikan jawaban secara singkat dan jelas.
 
Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>
 
{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
 
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [31]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": compression_retriever, "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

In [32]:
def ask_question(query):
    print(f"Question: {query}\n")
    
    response = rag_chain.invoke(query)
    
    print(f"Answer:")
    print(response)
    
    docs = retriever.invoke(query)
    print(f"\nSumber informasi:")
    for i, doc in enumerate(docs):
        print(f"- {i+1}. Halaman {doc.metadata.get('page', '?')}")

In [33]:
query = "Apa sih itu GenAI detector"
ask_question(query)

Question: Apa sih itu GenAI detector

Answer:
GenAI Detector adalah sebuah alat yang digunakan untuk mendeteksi apakah sebuah teks atau karya seni telah dibuat oleh generative artificial intelligence (Generative AI) atau tidak. Alat ini dapat mendiagnosa apakah teks atau karya seni tersebut telah dihasilkan oleh AI atau tidak, sehingga dapat membantu dalam mencegah plagiat dan memastikan integritas akademik.

Beberapa contoh alat GenAI Detector yang populer adalah:

* Turnitin AI Detector
* GPTZero
* Originality.ai

Alat-alat ini biasanya menggunakan teknik-teknik seperti analisis linguistik, analisis struktur kalimat, dan analisis konteks untuk mendeteksi adanya penggunaan AI dalam pembuatan teks atau karya seni.

Sumber informasi:
- 1. Halaman 8
- 2. Halaman 31
- 3. Halaman 25
- 4. Halaman 22
- 5. Halaman 17
